# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
<https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json>

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print("\nDATASET METADATA OVERVIEW:")
print(f"Name: {metadata.name}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Description: {metadata.description[:200]}...") # Print the first 200 characters for brevity

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields using @id references
print("RECORD SETS AVAILABLE IN DATASET:\n")
for rs in dataset.record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} (name: {field.name}, type: {field.data_type})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - @id: {col.id} (name: {col.name})")
    print('-'*40)

# For demonstration, pick the first record set for further exploration
record_set_ids = [rs.id for rs in dataset.record_sets]
if record_set_ids:
    example_record_set = record_set_ids[0]
    print(f"\nFirst record set @id for analysis: {example_record_set}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id`.

We'll extract all available record sets for demonstration.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set in dataset.record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"Loaded {len(df)} rows for record set @id: {record_set.id}")

# Show available dataframes (by record set @id)
print(f"\nDataFrames keyed by record set @id: {list(dataframes.keys())}")

if dataframes:
    target_record_set_id = list(dataframes.keys())[0]  # Use the first for demo
    print(f"\nColumns in {target_record_set_id}:\n{dataframes[target_record_set_id].columns.tolist()}")
    display(dataframes[target_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
import numpy as np

# List of numeric fields by field @id in chosen record set for demonstration
target_record_set = None
for rs in dataset.record_sets:
    if rs.id == target_record_set_id:
        target_record_set = rs
        break

numeric_field_id = None
for field in (target_record_set.fields if target_record_set else []):
    if field.data_type in ['Float', 'Integer', 'Number']:
        numeric_field_id = field.id
        print(f"Selected numeric field for EDA: @id: {numeric_field_id}, name: {field.name}")
        break

if numeric_field_id and numeric_field_id in dataframes[target_record_set_id].columns:
    threshold = 10
    print(f"\nFiltering records with {numeric_field_id} > {threshold}:")
    filtered_df = dataframes[target_record_set_id][dataframes[target_record_set_id][numeric_field_id] > threshold]
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst few rows with normalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Try grouping by a string/categorical field
    group_field_id = None
    for field in target_record_set.fields:
        # Try to find a non-numeric, non-ID field to group by
        if field.data_type == 'Text' and field.id != numeric_field_id and field.id in filtered_df.columns:
            group_field_id = field.id
            print(f"Grouping by: @id: {group_field_id}, name: {field.name}")
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in dataframes[target_record_set_id].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[target_record_set_id][numeric_field_id].dropna(), bins=18, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by group if possible
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, 
                    data=dataframes[target_record_set_id].dropna(subset=[numeric_field_id, group_field_id]))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No numeric field found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to explore a Croissant-based mental health survey dataset using `mlcroissant`. Using only entity `@id` references for all record sets and fields, we loaded data, performed basic EDA, and visualized distributions. For more advanced analysis, consider investigating additional record sets, handling missing data, and leveraging the rich schema metadata further!